# Terceira lista de simulação: agrupamento de dados
## Base de dados: Dígitos (`load_digits`) — algoritmos: Hierárquico, K-Means, GMM e DBSCAN

### Preparação dos dados e funções auxiliares

In [ ]:
# Matriz de características (features) e vetor de objetivos (target) do dataset Números

X = digits.data
y = digits.target
print(X.shape)
print(y.shape)

In [ ]:
# Importação das métricas de avaliação de desempenho de algoritmos de agrupamento
from sklearn.metrics.cluster import adjusted_rand_score
from sklearn.metrics.cluster import adjusted_mutual_info_score
from sklearn.metrics.cluster import homogeneity_score
from sklearn.metrics.cluster import completeness_score
from sklearn.metrics.cluster import silhouette_score

# Função auxiliar: imprime as métricas no mesmo formato utilizado no notebook
def avalia(nome, X, y, clusters):
  print("{} - ARI: {:.2f}".format(nome, adjusted_rand_score(y, clusters)))
  print("{} - MIS: {:.2f}".format(nome, adjusted_mutual_info_score(y, clusters)))
  print("{} - Homogeneity: {:.2f}".format(nome, homogeneity_score(y, clusters)))
  print("{} - Completeness: {:.2f}".format(nome, completeness_score(y, clusters)))
  if len(set(clusters)) > 1:      # o Silhouette só é definido para 2 ou mais grupos
    print("{} - Silhouette: {:.2f}".format(nome, silhouette_score(X, clusters)))
  else:
    print("{} - Silhouette: não definido (apenas 1 grupo)".format(nome))
  print()

# Redução para 2 dimensões (PCA) APENAS para visualização, já que os dígitos têm 64 features
from sklearn.decomposition import PCA
X_pca = PCA(n_components=2).fit_transform(X)

# Visualização dos grupos reais (verdade de solo)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y, s=40, cmap='viridis')
plt.colorbar()

## 1. Algoritmo Hierárquico Aglomerativo

In [ ]:
# Aplicação do algoritmo de agrupamento - hierárquico aglomerativo - dataset Números

from sklearn.cluster import AgglomerativeClustering              # 1. Importação da classe do modelo a ser utilizado

hierarquico = AgglomerativeClustering(n_clusters = 10)            # 2. Criação do objeto do modelo desejado

y_h = hierarquico.fit_predict(X)                                  # 3. Ajuste do modelo, com o dado de treino

# Visualização dos resultados
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_h, s=40, cmap='viridis')

In [ ]:
# Hiperparâmetros do modelo Hierárquico Aglomerativo
hierarquico

In [ ]:
# Métricas de desempenho - Hierárquico Aglomerativo (linkage = 'ward', n_clusters = 10)
avalia("AgglomerativeClustering", X, y, y_h)

In [ ]:
# Influência do hiperparâmetro linkage (n_clusters = 10)

fig, axes = plt.subplots(1, 4, figsize=(16, 4), subplot_kw={'xticks': (), 'yticks': ()})

for ax, linkage in zip(axes, ('ward', 'complete', 'average', 'single')):
  clustering = AgglomerativeClustering(linkage = linkage, n_clusters = 10)
  clusters = clustering.fit_predict(X)
  ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, s=40, cmap='viridis')
  ax.set_title(linkage + " linkage", size = 15)
  avalia("linkage = " + linkage, X, y, clusters)

In [ ]:
# Influência do hiperparâmetro n_clusters (linkage = 'ward')

for n in (2, 5, 8, 10, 12, 15, 20):
  clusters = AgglomerativeClustering(n_clusters = n).fit_predict(X)
  avalia("n_clusters = {}".format(n), X, y, clusters)

### Análise - Hierárquico Aglomerativo
**Desempenho (ward, 10 clusters):** ARI = 0.79, MIS = 0.87, Homogeneity = 0.86, Completeness = 0.88 e Silhouette = 0.18. É o melhor resultado entre os algoritmos testados no dataset Números.

**Influência do `linkage`:** o critério de ligação foi o hiperparâmetro de maior impacto. O `ward` (que minimiza o aumento da variância intra-cluster, favorecendo grupos compactos) obteve ARI = 0.79; `average` ficou em 0.51 e `complete` em 0.43. O `single` colapsou (ARI = 0.00 e Silhouette negativo = -0.13): por conectar grupos por um único par de pontos próximos (efeito de encadeamento), ele cria um cluster gigante e vários clusters minúsculos/outliers.

**Influência do `n_clusters`:** com poucos clusters (2 e 5) a Completeness é alta (0.97 e 0.88), mas a Homogeneity é baixa (0.26 e 0.59), pois classes distintas são fundidas no mesmo grupo. Com muitos clusters ocorre o oposto: a Homogeneity sobe (0.93 com 20 grupos) e a Completeness cai (0.74), porque uma mesma classe é dividida em vários grupos. O melhor equilíbrio (ARI) ocorre em torno de 10-12 clusters, coerente com as 10 classes reais.

**Silhouette:** o valor é baixo (~0.18) em todos os casos. Isso mostra que, no espaço de 64 dimensões, os grupos não são bem separados (as classes se sobrepõem), embora o ARI/MIS indiquem boa concordância com as classes reais. Ou seja, Silhouette baixo não significa agrupamento ruim quando a estrutura real é sobreposta.


## 2. Algoritmo K-Means

In [ ]:
# Etapas do treiamento não-supervisionado - clusterização - dataset Números
from sklearn.cluster import KMeans         # 1. Importação da classe do modelo a ser utilizado

kmeans = KMeans(n_clusters=10, random_state=0)   # 2. Criação do objeto do modelo desejado
kmeans.fit(X)                                    # 3. Ajuste do modelo, com o dado de treino
y_kmeans = kmeans.predict(X)                     # 4. Predição dos grupos feito pelo modelo treinado para um deternimado conjunto de dados

# Visualização dos resultados

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_kmeans, s=40, cmap='viridis')

centers = PCA(n_components=2).fit(X).transform(kmeans.cluster_centers_)   # centróides projetados no plano do PCA
plt.scatter(centers[:, 0], centers[:, 1], c='black', s=200, alpha=0.5)

In [ ]:
# Hiperparâmetros do modelo Kmeans
kmeans

In [ ]:
# Centróides (cluster_centers_) exibidos como imagens 8x8 - cada centróide é o "dígito médio" do grupo
fig, ax = plt.subplots(2, 5, figsize=(8, 3), subplot_kw={'xticks':[], 'yticks':[]})
centers_img = kmeans.cluster_centers_.reshape(10, 8, 8)
for axi, center in zip(ax.flat, centers_img):
  axi.imshow(center, interpolation='nearest', cmap=plt.cm.binary)

In [ ]:
# Métricas de desempenho - K-Means (n_clusters = 10)
avalia("KMeans", X, y, y_kmeans)

In [ ]:
# Influência do hiperparâmetro n_clusters

for n in (5, 8, 10, 12, 15):
  clusters = KMeans(n_clusters=n, random_state=0).fit_predict(X)
  avalia("n_clusters = {}".format(n), X, y, clusters)

In [ ]:
# Influência do hiperparâmetro max_iter (n_init=1 para isolar o efeito do número de iterações)

for mi in (1, 2, 5, 7, 10):
  clusters = KMeans(n_clusters=10, max_iter=mi, n_init=1, random_state=0).fit_predict(X)
  avalia("max_iter = {}".format(mi), X, y, clusters)

In [ ]:
# Influência do hiperparâmetro init (iniciação dos centros): k-means++ x random

for init in ('k-means++', 'random'):
  clusters = KMeans(n_clusters=10, init=init, n_init=10, random_state=0).fit_predict(X)
  avalia("init = " + init, X, y, clusters)

### Análise - K-Means
**Desempenho (10 clusters):** ARI = 0.62, MIS = 0.73, Homogeneity = 0.72, Completeness = 0.75 e Silhouette = 0.19. Resultado inferior ao hierárquico com ward, embora com o maior Silhouette (o K-Means otimiza diretamente a compacidade dos grupos, que é o que o Silhouette mede). Os centróides mostrados como imagens 8x8 representam o "dígito médio" de cada grupo (comente na sua execução quais dígitos ficaram bem representados).

**Influência do `n_clusters`:** com 5 e 8 grupos o desempenho cai bastante (ARI = 0.32 e 0.51), pois classes diferentes são fundidas. O pico ocorre em k = 12 (ARI = 0.65), e com k = 15 o ARI diminui (0.58) enquanto a Homogeneity continua subindo e a Completeness cai (classes divididas em vários grupos). O Silhouette varia pouco (0.13 a 0.19), portanto não é um bom critério isolado para escolher k nesta base.

**Influência do `max_iter`:** praticamente nenhuma. Mesmo com 1 iteração o ARI foi 0.62, e entre 2 e 10 iterações as métricas variam apenas ±0.02. Isso indica que, com a inicialização k-means++, o algoritmo já parte de centros bons e converge muito rápido nesta base.

**Influência do `init`:** `k-means++` (ARI = 0.67) e `random` (ARI = 0.66) tiveram resultados parecidos quando se usa `n_init=10` (escolhe-se a melhor de 10 execuções). Lembrar que o K-Means é sensível à iniciação dos centros, por isso a diferença entre execuções com `random_state` distintos pode ser maior que a diferença entre esses dois métodos.


## 3. Algoritmo GMM - Gaussian Mixture Models

In [ ]:
# Importação, instanciação e uso do modelo GMM - dataset Números

from sklearn.mixture import GaussianMixture                                  # 1. Importação do modelo
gmm = GaussianMixture(n_components=10, covariance_type='full', random_state=0).fit(X)   # 2+3.Instanciação e treino do modelo
labels_gmm = gmm.predict(X)                                                  # 4. Predição do modelo

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_gmm, s=40, cmap='viridis');

In [ ]:
# Hiperparâmetros do modelo GMM
gmm

In [ ]:
# Matriz de probilidade
probs = gmm.predict_proba(X)
print(probs[:5].round(3))

# o raio dos ponto são proporcionais a probabilidade do cluster mais representativo desse ponto
size = 50 * probs.max(1) ** 2
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_gmm, cmap='viridis', s=size);

In [ ]:
# Métricas de desempenho - GMM (n_components = 10, covariance_type = 'full')
avalia("GaussianMixture", X, y, labels_gmm)

In [ ]:
# Influência do hiperparâmetro covariance_type (n_components = 10)

for ct in ('full', 'tied', 'diag', 'spherical'):
  clusters = GaussianMixture(n_components=10, covariance_type=ct, random_state=0).fit_predict(X)
  avalia("covariance_type = " + ct, X, y, clusters)

In [ ]:
# Influência do hiperparâmetro n_components (covariance_type = 'full')

for n in (5, 8, 10, 12, 15):
  clusters = GaussianMixture(n_components=n, covariance_type='full', random_state=0).fit_predict(X)
  avalia("n_components = {}".format(n), X, y, clusters)

### Análise - GMM
**Desempenho (10 componentes, covariância `full`):** ARI = 0.61, MIS = 0.73, Homogeneity = 0.72, Completeness = 0.74 e Silhouette = 0.17, muito próximo do K-Means. A matriz de probabilidades (`predict_proba`) mostra que a maioria das amostras é atribuída a um cluster com probabilidade ~1, o que é típico em dados de alta dimensão (64 features), onde as gaussianas ficam muito "afiadas".

**Influência do `covariance_type`:** `tied` (uma covariância compartilhada) teve o melhor resultado (ARI = 0.63, MIS = 0.76), seguido de `full` (0.61) e `spherical` (0.59); `diag` foi o pior (ARI = 0.52). O `full` estima 10 matrizes 64x64 com apenas 1797 amostras, então tem muitos parâmetros e tende ao sobreajuste, enquanto o `tied` é mais restrito e mais estável. O `diag` assume features independentes, o que não é verdade para pixels vizinhos, e por isso piora.

**Influência do `n_components`:** com 5 e 8 componentes o ARI cai (0.32 e 0.51); com 10 e 12 se mantém em 0.61 e com 15 volta a cair (0.51), a mesma tendência de fusão/divisão de classes vista nos outros algoritmos.

**Observação:** o GMM depende da inicialização (algoritmo EM), por isso foi fixado `random_state=0` para reprodutibilidade.


## 4. Algoritmo DBSCAN

In [ ]:
# Importação, instanciação e uso do modelo DBSCAN - dataset Números

from sklearn.cluster import DBSCAN                               # 1. Importação do modelo
dbscan = DBSCAN()                                                # 2. Instanciação do modelo (eps=0.5 e min_samples=5 - valores default)
labels_dbscan = dbscan.fit_predict(X)                            # 3+4. Treino e predição do modelo

print("Cluster memberships:\n{}".format(labels_dbscan))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_dbscan, s=40, cmap='viridis');

In [ ]:
# Hiperparâmetros do modelo DBSCAN
dbscan

In [ ]:
# Influência dos parâmetros eps e min_samples (dados originais, sem normalização)
# Obs.: o rótulo -1 representa ruído (pontos que não pertencem a nenhum cluster)

for eps in (10, 15, 20, 25, 30):
  for ms in (5, 10):
    clusters = DBSCAN(eps=eps, min_samples=ms).fit_predict(X)
    print("eps = {}, min_samples = {} -> {} clusters, {} pontos de ruído".format(
        eps, ms, len(set(clusters)) - (1 if -1 in clusters else 0), np.sum(clusters == -1)))
    avalia("DBSCAN(eps={}, min_samples={})".format(eps, ms), X, y, clusters)

In [ ]:
# Melhor configuração encontrada: eps=20 e min_samples=5
dbscan = DBSCAN(eps=20, min_samples=5)
labels_dbscan = dbscan.fit_predict(X)

plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels_dbscan, s=40, cmap='viridis');
avalia("DBSCAN(eps=20, min_samples=5)", X, y, labels_dbscan)

In [ ]:
# Influência da normalização das features (reescalonamento com média zero e variância um)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(X)
X_scaled = scaler.transform(X)

for eps in (3, 5, 7, 10):
  clusters = DBSCAN(eps=eps, min_samples=5).fit_predict(X_scaled)
  print("eps = {} -> {} clusters, {} pontos de ruído".format(
      eps, len(set(clusters)) - (1 if -1 in clusters else 0), np.sum(clusters == -1)))
  avalia("DBSCAN normalizado (eps={})".format(eps), X_scaled, y, clusters)

### Análise - DBSCAN
**Desempenho com os parâmetros default (`eps=0.5`, `min_samples=5`):** todos os pontos foram classificados como ruído (rótulo -1). Em 64 dimensões as distâncias entre amostras são muito maiores que 0.5, então nenhuma região atinge a densidade necessária.

**Influência de `eps` e `min_samples`:** o `eps` precisa ser da ordem de grandeza das distâncias entre vizinhos (~20 para esta base).
- `eps` ≤ 10: nenhum cluster (100% ruído). `eps` = 15: muito ruído (1553 pontos com `min_samples=5`).
- `eps` = 20 e `min_samples` = 5: melhor resultado (ARI = 0.47, MIS = 0.71, Homogeneity = 0.75), com 25 clusters e 464 pontos de ruído.
- `eps` = 25: os grupos começam a se unir (4 a 6 clusters, ARI ≤ 0.11); `eps` = 30: tudo vira um único cluster.
- Aumentar `min_samples` de 5 para 10 exige mais densidade: o ruído sobe (464 → 827 em `eps` = 20) e o ARI cai (0.47 → 0.25).

O DBSCAN é muito sensível a `eps`: uma faixa estreita separa "tudo é ruído" de "tudo é um único cluster" (a chamada maldição da dimensionalidade para métodos baseados em densidade).

**Influência da normalização:** diferente do exemplo das luas, aqui a normalização com `StandardScaler` **piorou** o resultado (ARI ≈ 0.00 em todos os `eps` testados). Uma provável causa é que pixels de borda são quase sempre zero (variância ínfima) e, ao padronizá-los, pequenas variações são amplificadas, distorcendo as distâncias. Os Silhouettes aparentemente altos (0.27 e 0.64 em `eps` = 7 e 10) são enganosos: ocorrem quando quase tudo está num único cluster.

**Observação sobre as métricas:** o ruído (-1) é tratado pelas métricas como se fosse um cluster. Por isso o Silhouette do DBSCAN (0.07) e as demais métricas devem ser interpretados com cuidado, e o número de pontos de ruído deve ser reportado junto. O DBSCAN também **não** exige definir o número de clusters, mas aqui encontrou 25 em vez de 10, fragmentando as classes e deixando 26% das amostras como ruído.


## 5. Comparação final dos quatro algoritmos

In [ ]:
# Comparação dos algoritmos de agrupamento no dataset Números

from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN

fig, axes = plt.subplots(1, 5, figsize=(20, 4), subplot_kw={'xticks': (), 'yticks': ()})

# Lista dos algoritmos utilizados (melhores hiperparâmetros encontrados)
algorithms = [AgglomerativeClustering(n_clusters=10, linkage='ward'), KMeans(n_clusters=10, random_state=0),
              GaussianMixture(n_components=10, covariance_type='tied', random_state=0), DBSCAN(eps=20, min_samples=5)]

# plot dos grupos reais
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y, cmap='viridis', s=30)
axes[0].set_title("Grupos reais (target)")

# plot dos agrupamentos pelos algoritmos: Hierárquico Aglomerativo, K-Means, GMM e DBSCAN
for ax, algorithm in zip(axes[1:], algorithms):
  clusters = algorithm.fit_predict(X)
  ax.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', s=30)
  ax.set_title("{} - ARI: {:.2f}".format(algorithm.__class__.__name__, adjusted_rand_score(y, clusters)))

In [ ]:
# Métricas de avaliação de desempenho dos algoritmos de agrupamento

for algorithm in algorithms:
  clusters = algorithm.fit_predict(X)
  avalia(algorithm.__class__.__name__, X, y, clusters)

### Conclusão comparativa
| Algoritmo | ARI | MIS | Homogeneity | Completeness | Silhouette |
|---|---|---|---|---|---|
| Hierárquico (ward, 10) | **0.79** | **0.87** | **0.86** | **0.88** | 0.18 |
| K-Means (k=10) | 0.62 | 0.73 | 0.72 | 0.75 | **0.19** |
| GMM (tied, 10) | 0.63 | 0.76 | 0.75 | 0.78 | 0.18 |
| DBSCAN (eps=20, ms=5) | 0.47 | 0.71 | 0.75 | 0.68 | 0.07 |

- **Melhor algoritmo:** o Hierárquico com `linkage='ward'` foi o melhor em todas as métricas com conhecimento a priori (ARI, MIS, Homogeneity, Completeness). K-Means e GMM ficaram empatados em desempenho intermediário (ARI ≈ 0.62), e o DBSCAN foi o pior e o mais difícil de ajustar.
- **Hiperparâmetros:** o mais crítico em cada algoritmo foi: `linkage` (Hierárquico), `n_clusters` (K-Means), `covariance_type` e `n_components` (GMM), e `eps` (DBSCAN). `max_iter` e `init` tiveram impacto desprezível no K-Means.
- **Métricas:** ARI e MIS (que usam os rótulos reais) discriminam bem os algoritmos. O Silhouette ficou entre 0.07 e 0.19 para todos, não diferenciou bem os métodos, e chegou a favorecer o K-Means mesmo ele sendo pior que o hierárquico (Silhouette mede compacidade geométrica, não concordância com as classes). Homogeneity e Completeness ajudam a entender *como* o algoritmo erra: poucos clusters aumentam a Completeness e muitos clusters aumentam a Homogeneity.
- **Visualização:** os gráficos usam PCA com 2 componentes apenas para visualização; como a projeção perde muita informação dos 64 pixels, as classes parecem sobrepostas no plano mesmo quando os algoritmos as separam melhor no espaço original.
